# CREM: Tamaño Medio del Hogar y Porcentaje de Hogares Unipersonales
**Propósito**: Extrae la información sobre el tamaño medio del hogar y el porcentaje de hogares unipersonales por municipios a partir de la página oficial del CREM.

**Particularidades de esta tabla**:
- La cabecera tiene **dos filas**: la primera agrupa sub-tablas por nombre (con `colspan`), la segunda contiene los años.
- El nombre de columna resultante es `{año}_{nombre_subtabla}` normalizado.
- Se filtran únicamente las filas de nivel **Municipio** (`level2`).

In [1]:
import sys
import re
from bs4 import BeautifulSoup
import pandas as pd
from pyspark.sql import functions as F

sys.path.insert(0, '/tfm/python_notebooks')
from tfm_lib import utils as tfm_utils
from tfm_lib import crem_scrap as crem

In [2]:
url_base   = "https://econet.carm.es/web/crem/inicio/-/crem/sicrem/PM2089//sec31.html"
table_name = "raw.crem.tamano_medio_hogar_y_pct_hogares_unipersonales"

## Descarga y Parseo del Contenido HTML directamente en Memoria

In [3]:
html = crem.get_html(url_base)
soup = BeautifulSoup(html, "html.parser")

table = soup.find("table", id="interiorTablaDatos")
if not table:
    raise Exception("No se encontró la tabla con id 'interiorTablaDatos' en el HTML")

# ---------------------------------------------------------------------------
# Parseo de cabecera de dos filas:
#   Fila 0: grupos con colspan  → nombre del grupo (sub-tabla)
#   Fila 1: años                → valor de cada columna de datos
# Resultado: lista de etiquetas compuestas  "{año}_{Grupo}"
# ---------------------------------------------------------------------------
thead    = table.find("thead", id="cabeceraTablaDatos") or table.find("thead")
hdr_rows = thead.find_all("tr")

# Fila 0: grupos con colspan
grupos = []
for th in hdr_rows[0].find_all("th"):
    texto = th.get_text(strip=True)
    if not texto:
        continue
    span = int(th.get("colspan", 1))
    grupos.extend([texto] * span)

# Fila 1: años
anios = [th.get_text(strip=True) for th in hdr_rows[1].find_all("th") if th.get_text(strip=True)]

columnas = [f"{a}_{g}" for a, g in zip(anios, grupos)]
print(f"Columnas detectadas ({len(columnas)}): {columnas[:6]} …")

Columnas detectadas (18): ['2023_Tamaño medio del hogar', '2022_Tamaño medio del hogar', '2021_Tamaño medio del hogar', '2020_Tamaño medio del hogar', '2019_Tamaño medio del hogar', '2018_Tamaño medio del hogar'] …


In [4]:
# ---------------------------------------------------------------------------
# Parseo de filas: solo level2 (Municipio)
# ---------------------------------------------------------------------------
tbody = table.find("tbody")

mapping_niveles = {1: "Región", 2: "Municipio", 3: "Distrito", 4: "Sección"}

datos_filas = []
for tr in tbody.find_all("tr"):
    th = tr.find("th", id="r")
    if not th:
        continue

    # Detectar nivel
    ul = th.find("ul")
    level_num = None
    if ul and ul.has_attr("class"):
        for c in ul["class"]:
            if c.startswith("level"):
                try:
                    level_num = int(c.replace("level", ""))
                except ValueError:
                    pass
                break

    # Solo Municipios
    if mapping_niveles.get(level_num) != "Municipio":
        continue

    label_clean = crem.clean_and_decode(th.get_text(strip=True))
    match = re.match(r"^(.*?)\s*-\s*(\d+)$", label_clean)
    nombre = match.group(1).strip() if match else label_clean
    if ',' in nombre:
        nombre = nombre.split(',')[1].strip() + ' ' + nombre.split(',')[0].strip()

    tds    = tr.find_all("td", id="d", class_="tbdato")
    values = []
    for td in tds:
        val_str = td.get_text(strip=True).replace(",", ".")
        try:
            values.append(float(val_str) if val_str not in ("", "-", "..") else None)
        except ValueError:
            values.append(None)

    if len(values) == len(columnas):
        registro = {"nombre": crem.normalize_name(nombre)}
        for col, val in zip(columnas, values):
            registro[col] = val
        datos_filas.append(registro)

print(f"Municipios parseados: {len(datos_filas)}")

Municipios parseados: 45


## Integración con PySpark y Delta Lake

In [5]:
spark = tfm_utils.get_spark_session(app_name="CREM_Tamano_Hogar_Unipersonales")

df = spark.createDataFrame(datos_filas)

# Normalizar columnas (minúsculas, sin acentos)
df_normalized = tfm_utils.normalize_df(df)
tfm_utils.display(df_normalized)

,2015_porcentaje_de_hogares_unipersonales,2015_tamano_medio_del_hogar,2016_porcentaje_de_hogares_unipersonales,2016_tamano_medio_del_hogar,2017_porcentaje_de_hogares_unipersonales,2017_tamano_medio_del_hogar,2018_porcentaje_de_hogares_unipersonales,2018_tamano_medio_del_hogar,2019_porcentaje_de_hogares_unipersonales,2019_tamano_medio_del_hogar,2020_porcentaje_de_hogares_unipersonales,2020_tamano_medio_del_hogar,2021_porcentaje_de_hogares_unipersonales,2021_tamano_medio_del_hogar,2022_porcentaje_de_hogares_unipersonales,2022_tamano_medio_del_hogar,2023_porcentaje_de_hogares_unipersonales,2023_tamano_medio_del_hogar,nombre
0,28.9,2.5,27.8,2.5,29.9,2.5,29.0,2.5,29.3,2.5,29.3,2.5,30.3,2.4,31.5,2.4,31.9,2.4,abanilla
1,22.4,2.8,22.1,2.8,23.0,2.8,23.1,2.7,23.3,2.7,23.0,2.7,23.9,2.7,23.9,2.7,24.9,2.7,abaran
2,26.0,2.7,25.5,2.8,26.2,2.7,25.8,2.7,25.9,2.7,26.4,2.7,27.2,2.7,27.1,2.7,27.1,2.7,aguilas
3,29.5,2.6,29.5,2.6,29.7,2.6,28.1,2.6,28.1,2.6,27.9,2.6,29.3,2.6,31.3,2.5,33.4,2.5,albudeite
4,22.3,2.8,21.8,2.8,22.5,2.8,22.5,2.8,22.4,2.8,22.4,2.8,22.8,2.8,22.8,2.8,22.9,2.8,alcantarilla
5,29.1,2.5,25.8,2.6,28.2,2.6,28.0,2.6,29.5,2.5,28.0,2.5,28.2,2.6,27.6,2.6,27.4,2.5,aledo
6,20.0,3.1,19.3,3.0,19.6,3.0,20.1,3.0,20.2,3.0,20.9,3.0,20.4,3.0,20.2,3.1,21.5,3.0,alguazas
7,26.2,2.7,25.1,2.7,26.2,2.7,26.0,2.7,26.0,2.7,27.0,2.7,27.0,2.7,26.9,2.7,27.1,2.7,alhama_de_murcia
8,22.3,2.9,20.8,2.9,21.3,2.9,21.7,2.9,22.2,2.9,22.6,2.9,22.9,2.8,22.9,2.9,22.7,2.9,archena
9,17.8,3.2,17.0,3.2,17.6,3.2,18.2,3.1,18.5,3.2,18.7,3.1,18.9,3.1,19.0,3.1,19.5,3.1,beniel


In [6]:
delta_path = tfm_utils.full_table_path(table_name)
print(f"Escribiendo {df_normalized.count()} registros en: {table_name}")
print(f"Destino: {delta_path}")

(df_normalized
    .write
    .mode("overwrite")
    .format("delta")
    .save(delta_path)
)
print("¡Escritura en Delta Lake completada con éxito!")

Escribiendo 45 registros en: raw.crem.tamano_medio_hogar_y_pct_hogares_unipersonales
Destino: /tfm/delta_lake/raw/crem/tamano_medio_hogar_y_pct_hogares_unipersonales
¡Escritura en Delta Lake completada con éxito!
